In [6]:
# 测试 PolyGA 中 PolyNation.__crossover 的行为

from polyga.polygod import PolyPlanet, PolyLand, PolyNation
from polyga import utils

import pandas as pd

# 1. 构造一个最小的 PolyPlanet
# 预测和指纹函数在这里用占位实现即可，因为本测试只关注交叉逻辑
planet = PolyPlanet(
    name="TestPlanet",
    predict_function=lambda df, fp_headers, models: df,  # 不改动 df
    fingerprint_function=lambda df: (df, []),           # 不增加指纹列
    models=None,
    num_cpus=1,
    random_seed=123,
)

# 2. 构造 PolyLand，使用真实的生成函数 chromosome_ids_to_smiles
land = PolyLand(
    name="TestLand",
    planet=planet,
    generative_function=utils.chromosome_ids_to_smiles,
    fitness_function=lambda df, fp_headers: df.assign(fitness=1.0),
    crossover_position="relative_center",
)

# 3. 构造 PolyNation，自动生成随机初始种群
nation = PolyNation(
    name="TestNation",
    land=land,
    num_population_initial=6,       # 小种群，便于观察
    num_chromosomes_initial=6,      # 每条链的 block 数
    num_families=2,
    num_parents_per_family=3,
    num_children_per_family=4,
    selection_scheme=lambda pop, quota: pop,  # 直接全部作为可选父代
    partner_selection="random",             # 不依赖指纹
    random_seed=123,
)

# 查看初始种群的 chromosome_ids
print("Initial population chromosome_ids (planetary_id -> chromosome_ids):")
for _, row in nation.population.iterrows():
    print(row["planetary_id"], "->", row["chromosome_ids"])

# 4. 手动构造一个 family，避免依赖内部选择逻辑
# 这里只取前 3 个个体作为一组 family
parent_ids = nation.population["planetary_id"].values.tolist()
family = parent_ids[:3]
print("\nFamily (parent planetary_ids):", family)

families = [family]

# 5. 通过名称改写规则调用私有方法 __crossover
children_chrom_ids, parents = nation._PolyNation__crossover(families)

print("\nChildren chromosome_ids and their parents:")
for child_ids, (p1, p2) in zip(children_chrom_ids, parents):
    print(f"parents=({p1}, {p2}) -> child_chromosome_ids={child_ids}")


Initial population chromosome_ids (planetary_id -> chromosome_ids):
1 -> [3, 136, 118, 10, 181, 44]
2 -> [90, 184, 89, 55, 157, 163]
3 -> [48, 164, 158, 42, 81, 148]
4 -> [167, 159, 42, 103, 158, 46]
5 -> [83, 36, 72, 2, 28, 94]
6 -> [103, 183, 123, 172, 92, 43]

Family (parent planetary_ids): [1, 2, 3]

Children chromosome_ids and their parents:
parents=(1, 2) -> child_chromosome_ids=[10, 181, 44, 90, 184, 89, 55]
parents=(1, 3) -> child_chromosome_ids=[10, 181, 44, 48, 164, 158]
parents=(2, 3) -> child_chromosome_ids=[157, 163, 48, 164, 158]
parents=(1, 2) -> child_chromosome_ids=[3, 136, 118, 90, 184, 89, 55]
